# Download entirety of the Datasets

In [9]:
import os
import numpy as np
import pandas as pd
import requests
import yfinance as yf

In [10]:
start_date = "2010-01-04"
end_date = "2024-12-31"
output_dir = "../data/sp500_individual/"
os.makedirs(output_dir, exist_ok=True)

In [11]:
def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    
    # Define a header to look like a browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    }
    
    # Use requests to get the HTML content with the header
    response = requests.get(url, headers=headers)
    
    # Pass the text content to pandas
    table = pd.read_html(response.text)
    df = table[0]
    
    # Replace dots with hyphens for yfinance compatibility
    tickers = df['Symbol'].str.replace('.', '-', regex=False).tolist()
    return tickers

In [12]:
def transformation(data):
        # YFinance indexes by Date automatically; ensure it is sorted
        data = data.sort_index()
        
        # Calculate log-returns: r_t = ln(P_t / P_{t-1}) [cite: 5353]
        ohlc_cols = ['Open', 'High', 'Low', 'Close']
        # Divide OHLC by previous day's close to get relative returns
        prev_close = data['Close'].shift(1)
        ohlc_log_rets = np.log(data[ohlc_cols].div(prev_close, axis=0))
        
        # Volume log-returns (adding 1 to avoid log(0))
        volume_log_rets = np.log(data['Volume'] + 1) - np.log(data['Volume'].shift(1) + 1)

        processed_data = pd.concat([ohlc_log_rets, volume_log_rets], axis=1).dropna()
        data_arr = processed_data.values.astype(np.float32)

        # Global Standardization: r_std = (r - mu) / sigma [cite: 5370]
        mean = data_arr.mean(axis=0)
        std = data_arr.std(axis=0)
        data_standardized = (data_arr - mean) / (std + 1e-8)
        
        return data_standardized

In [13]:
tickers = get_sp500_tickers()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15820\1754880765.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(response.text)


In [14]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BK',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'CPB',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 'CAG',
 'COP',
 'ED'

In [15]:
for ticker in tickers:
    try:
        df = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date,
            interval="1d",
            auto_adjust=True,
            progress=False
        )
        
        if df.empty:
            continue
            
        # Drop the Ticker level from columns if it exists (yfinance multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel('Ticker')

        # Check for completeness: 
        # Compare first/last index to target dates or check total row count
        # For the 2010-2024 period, you expect ~3770 trading days.
        actual_start = df.index[0].strftime('%Y-%m-%d')
        if actual_start > start_date:
            print(f"Skipping {ticker}: Data only starts at {actual_start}")
            continue

        # Reuse your transformation function
        data_standardized = transformation(df)
        
        # Save individually
        columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        df_processed = pd.DataFrame(data_standardized, columns=columns)
        
        file_path = os.path.join(output_dir, f"{ticker}_{start_date}_{end_date}_processed.csv")
        df_processed.to_csv(file_path, index=False)
        
    except Exception as e:
        print(f"Failed to process {ticker}: {e}")

Skipping ABBV: Data only starts at 2013-01-02
Skipping ABNB: Data only starts at 2020-12-10
Skipping ALLE: Data only starts at 2013-11-18
Skipping AMCR: Data only starts at 2012-05-15
Skipping APO: Data only starts at 2011-03-30
Skipping APP: Data only starts at 2021-04-15
Skipping APTV: Data only starts at 2011-11-17
Skipping ARES: Data only starts at 2014-05-02
Skipping ANET: Data only starts at 2014-06-06
Skipping XYZ: Data only starts at 2015-11-19
Skipping CARR: Data only starts at 2020-03-19
Skipping CVNA: Data only starts at 2017-04-28
Skipping CBOE: Data only starts at 2010-06-15
Skipping CDW: Data only starts at 2013-06-27
Skipping CHTR: Data only starts at 2010-01-05
Skipping CFG: Data only starts at 2014-09-24
Skipping COIN: Data only starts at 2021-04-14
Skipping CEG: Data only starts at 2022-01-19
Skipping CPAY: Data only starts at 2010-12-15
Skipping CTVA: Data only starts at 2019-05-24
Skipping CRWD: Data only starts at 2019-06-12
Skipping DDOG: Data only starts at 2019-


1 Failed download:
['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-04 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1262581200, endDate = 1735621200")')


Skipping HOOD: Data only starts at 2021-07-29



1 Failed download:
['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-04 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1262581200, endDate = 1735621200")')


Skipping NOW: Data only starts at 2012-06-29
Skipping SOLV: Data only starts at 2024-03-26
Skipping SYF: Data only starts at 2014-07-31
Skipping TRGP: Data only starts at 2010-12-07
Skipping TSLA: Data only starts at 2010-06-29
Skipping TTD: Data only starts at 2016-09-21
Skipping UBER: Data only starts at 2019-05-10
Skipping VLTO: Data only starts at 2023-10-04
Skipping VICI: Data only starts at 2018-01-02
Skipping VST: Data only starts at 2016-10-05
Skipping WDAY: Data only starts at 2012-10-12
Skipping XYL: Data only starts at 2011-10-13
Skipping ZTS: Data only starts at 2013-02-01
